# Copernicus Marine Service: Access Insitu Data 
* Need to have an account on Copernicus Marine
* The ~/.netrc file contains the account credentials: your login and password for [Copernicus Marine](https://marine.copernicus.eu/)
```
$ more ~/.netrc
machine auth.marine.copernicus.eu
  login xxxxxxxxxx 
  password xxxxxxxxxxxxx
```

In [ ]:
import copernicusmarine
import xarray as xr
import hvplot.xarray
import pandas as pd
from datetime import datetime, timezone, timedelta

In [ ]:
copernicusmarine.__version__

In [ ]:

var = 'SLEV'
ndays = 10   # get only last ndays of data

In [ ]:
# Get current UTC time
stop_time = datetime.now(timezone.utc)

# Subtract ndays
start_time = stop_time - timedelta(days=ndays)
print(f"start_time: {start_time}, stop_time={stop_time}")

## Find stations that have recent data

In [ ]:
%%script false --no-raise-error
# comment out the above to run; this just prevents inadvertant running since this takes a while
df_all = copernicusmarine.read_dataframe(
    dataset_id="cmems_obs-ins_med_phybgcwav_mynrt_na_irr",
    dataset_part="latest",
    variables=[var],
    start_datetime=stop_time - timedelta(hours=12),
    end_datetime=stop_time,
)
print(df_all['platform_id'].unique())

## Load data from a specific station

In [ ]:
sta = 'Taranto1TG'

In [ ]:
df = copernicusmarine.read_dataframe(
    dataset_id="cmems_obs-ins_med_phybgcwav_mynrt_na_irr",
    dataset_part="latest",
    variables=[var],
    start_datetime=start_time,
    end_datetime=stop_time,
    platform_ids=[sta]
)

In [ ]:
df['time'] = pd.to_datetime(df['time'])  # ensure it's datetime64[ns]

In [ ]:
ds = df.to_xarray()

In [ ]:
ds

In [ ]:
ds = df.set_index('time').to_xarray()
var = str(ds['variable'][0].data)
ds = ds.rename({'value':var})

In [ ]:
ds[var].hvplot(x='time', grid=True, title=f'Station {sta}')